# 05 — Disruption Prediction Model
**Module E**

Champion-challenger ensemble for 30-day stockout risk prediction:
- 20+ features spanning graph, detection, forecasting, and inventory layers
- XGBoost champion + LightGBM challenger
- Stacked logistic regression meta-learner
- SHAP TreeExplainer for feature attribution
- Metrics: PR-AUC, Precision@K, prediction lead time

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.features import build_feature_matrix, split_chronological
from src.models import (
    train_xgboost, train_lightgbm, build_ensemble, compute_shap_values
)
from src.viz import create_shap_summary, create_confusion_matrix_chart

## Build Feature Matrix

In [ ]:
feature_matrix = build_feature_matrix()
print(f'Feature matrix shape: {feature_matrix.shape}')
feature_matrix.head()

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, feature_names = split_chronological(feature_matrix)

## Train Models

In [ ]:
xgb_results = train_xgboost(X_train, y_train, X_val, y_val, feature_names)

In [ ]:
lgb_results = train_lightgbm(X_train, y_train, X_val, y_val, feature_names)

## Ensemble

In [ ]:
ensemble = build_ensemble(xgb_results, lgb_results, X_val, y_val, X_test, y_test)

## SHAP Analysis

In [ ]:
shap_results = compute_shap_values(xgb_results['model'], X_test, feature_names)

if shap_results['shap_values'] is not None:
    fig = create_shap_summary(shap_results['shap_values'], feature_names, shap_results['X_sample'])
    fig.show()

In [ ]:
# Confusion matrix
if 'confusion_matrix' in ensemble:
    fig_cm = create_confusion_matrix_chart(ensemble['confusion_matrix'])
    fig_cm.show()

## Performance Summary

In [ ]:
print('Model Performance Summary')
print('=' * 40)
print(f'XGBoost Val PR-AUC:   {xgb_results["val_prauc"]:.4f}')
print(f'LightGBM Val PR-AUC:  {lgb_results["val_prauc"]:.4f}')
print(f'Ensemble Val PR-AUC:  {ensemble["val_prauc"]:.4f}')
if 'test_prauc' in ensemble:
    print(f'Ensemble Test PR-AUC: {ensemble["test_prauc"]:.4f}')
    print(f'Precision@10%:        {ensemble.get("precision_at_k", 0):.4f}')